# 02 · Station map

Where are our stations, and where are trains late most often?

Steps:
1. Load `config/stations.csv` and `config/train_type_groups.csv`
2. Count arrivals and late arrivals (≥ 6 min) per EVA number **and train type** for one month
3. A function that filters by train types and merges the EVA numbers of one station into one point
4. Interactive map: choose a group, then the train types inside it

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import plotly.express as px
import polars as pl
from IPython.display import display

DATA_DIR = Path("../data/raw/hf/monthly_processed_data")
MONTH = "2026-09"  # change me: any downloaded month
MIN_STOPS = 30  # hide stations with fewer arrivals: 1 late train out of 1 would be "100 % late"

## 1. Stations and train type groups

The 12 bus stations have no StaDa entry, so they have no coordinates and cannot be drawn.

`train_type` in the data is often an operator brand (`NX`, `VIA`, `HLB`), not a product. `train_type_groups.csv` maps each type to `long_distance`, `regional`, `s_bahn` or `exclude`. Types that are not in the file are rare typos and replacement buses; they become `unmapped`.

In [ ]:
stations = pl.read_csv("../config/stations.csv", schema_overrides={"eva": pl.String})
groups = pl.read_csv("../config/train_type_groups.csv").select("train_type", "group")

print(stations.height, "EVA numbers,", stations["collect"].sum(), "with StaDa data")
groups.group_by("group").len()

## 2. Arrivals per EVA number and train type

Same label as in notebook 01: arrival delay computed from the arrival columns, cancelled stops and first stops of a ride (no arrival) removed.

We aggregate once to (EVA, train type). The result is small (about 1,100 rows), so the map can be recomputed instantly when you change the filter.

In [ ]:
counts = (
    # Lazily scan the monthly Parquet file (nothing is read yet, only a query plan is built)
    pl.scan_parquet(DATA_DIR / f"data-{MONTH}.parquet")
    # Keep only the 5 columns we need (projection pushdown: other columns are never read from disk)
    .select("eva", "train_type", "arrival_planned_time", "arrival_change_time", "arrival_is_canceled")
    .filter(
        # Only stations from our `stations` DataFrame; implode() turns the column into a single list value
        pl.col("eva").is_in(stations["eva"].implode()),
        # Only rows that actually have a planned arrival (drops departures-only rows, e.g. at origin stations)
        pl.col("arrival_planned_time").is_not_null(),
        # Exclude canceled arrivals (~ is logical NOT)
        ~pl.col("arrival_is_canceled"),
    )
    # Delay in whole minutes = actual/changed arrival minus planned arrival
    .with_columns(delay=(pl.col("arrival_change_time") - pl.col("arrival_planned_time")).dt.total_minutes())
    # One group per station AND train type (so we can filter by train type later)
    .group_by("eva", "train_type")
    # stops = number of arrivals; late_stops = number of arrivals with delay >= 6 minutes
    .agg(stops=pl.len(), late_stops=(pl.col("delay") >= 6).sum())
    # Execute the whole query plan and return an eager DataFrame
    .collect()
    # Add the group (long_distance, regional, ...) of each train type
    .join(groups, on="train_type", how="left")
    # Train types missing in train_type_groups.csv get the group "unmapped"
    .with_columns(pl.col("group").fill_null("unmapped"))
)
print(counts.height, "rows")
counts.group_by("group").agg(pl.col("stops").sum(), pl.col("train_type").n_unique().alias("types")).sort("stops", descending=True)

## 3. One point per station, for a chosen set of train types

Group by `stada_name`. We **sum** stops and late stops and divide afterwards. Averaging the late shares of the EVA numbers would give a small S-Bahn platform the same weight as the main hall.

Coordinates: those of the main EVA number (`is_main_eva`), otherwise the first one.

Stations with fewer than `MIN_STOPS` arrivals of the chosen types are dropped. Without this, a station with a single diverted ICE that was late shows up as "100 % late".

In [ ]:
def per_station(train_types: list[str]) -> pl.DataFrame:
    """Late share per station for the given train types."""
    per_eva = (
        counts.filter(pl.col("train_type").is_in(train_types))
        .group_by("eva")
        .agg(pl.col("stops").sum(), pl.col("late_stops").sum())
    )
    return (
        stations.filter(pl.col("collect"))
        .join(per_eva, on="eva", how="inner")
        .sort("is_main_eva", descending=True)  # main EVA first, so .first() picks its coordinates
        .group_by("stada_name")
        .agg(
            pl.col("federal_state").first(),
            pl.col("lat").first(),
            pl.col("lon").first(),
            pl.len().alias("n_evas"),
            pl.col("stops").sum(),
            pl.col("late_stops").sum(),
        )
        .filter(pl.col("stops") >= MIN_STOPS)
        .with_columns(late_share=(pl.col("late_stops") / pl.col("stops")).round(3))
        .sort("late_share", descending=True)
    )


per_station(["ICE"]).head(5)

## 4. Interactive map

- **Group:** `all` or one group. The list next to it then shows only the train types of that group, sorted by number of arrivals.
- **Train types:** Ctrl/Shift + click to select several. **Select all** selects every type of the group again.

Point size = number of arrivals, colour = late share. The colour scale is fixed from 0 % to 60 %, so maps of different groups can be compared.

In [ ]:
def types_of(group: str) -> list[str]:
    """Train types of a group, most arrivals first."""
    subset = counts if group == "all" else counts.filter(pl.col("group") == group)
    return subset.group_by("train_type").agg(pl.col("stops").sum()).sort("stops", descending=True)["train_type"].to_list()


def draw(train_types: list[str]) -> None:
    df = per_station(train_types)
    if df.is_empty():
        print("No station has enough arrivals for this selection.")
        return
    label = ", ".join(train_types) if len(train_types) <= 5 else f"{len(train_types)} train types"
    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        color="late_share",
        size="stops",
        hover_name="stada_name",
        hover_data={"federal_state": True, "stops": True, "late_share": ":.1%", "lat": False, "lon": False},
        color_continuous_scale="RdYlGn_r",  # green = on time, red = late
        range_color=(0, 0.6),
        size_max=25,
        zoom=4.8,
        center={"lat": 51.2, "lon": 10.4},
        map_style="carto-positron",
        title=f"Arrivals ≥ 6 min late, {MONTH} · {label} · {df['stops'].sum():,} arrivals at {df.height} stations",
        height=750,
    )
    fig.update_layout(coloraxis_colorbar_tickformat=".0%")
    fig.show()


group_choice = widgets.Dropdown(
    options=["all", "long_distance", "regional", "s_bahn", "exclude", "unmapped"],
    value="long_distance",
    description="Group",
)
type_choice = widgets.SelectMultiple(description="Train types", rows=10)
select_all = widgets.Button(description="Select all")
output = widgets.Output()


def on_group_change(change=None) -> None:
    types = types_of(group_choice.value)
    type_choice.options = types
    type_choice.value = tuple(types)  # select all types of the new group


def on_types_change(change=None) -> None:
    with output:
        output.clear_output(wait=True)
        draw(list(type_choice.value))


group_choice.observe(on_group_change, names="value")
type_choice.observe(on_types_change, names="value")
select_all.on_click(lambda _: setattr(type_choice, "value", tuple(type_choice.options)))

on_group_change()
display(widgets.HBox([widgets.VBox([group_choice, select_all]), type_choice]), output)

## Questions

1. Compare `long_distance` and `s_bahn`. Which group is later, and where?
2. In `regional`, select only `RE` and then only `NX`. Where does `NX` run? (Tip: notebook 01, `line_number`)
3. Look at `exclude` (buses). Why are almost all points green, and why is that misleading?
4. Change `MONTH` to a winter month (e.g. `2025-01`) and to `2024-07`. What changes?
5. Could "region" or "station size" be a useful feature? Which leakage rule would it have to follow?